# 🤖 INTELLIGENT IT/HR HELPDESK TRIAGE
## IT/HR Helpdesk Triage Agent

### 🗺️ How does the system work?
```
Employee Question
      ↓
Router Agent (Classifies the question)
      ↓              ↓
requires_escalation=False   requires_escalation=True
      ↓                           ↓
RAG searches Wiki          create_it_ticket() is called
      ↓                           ↓
Answer from documents      Formal ticket opened

*   List item
*   List item


```

### 📦 Components we will build:
1. **Setup** — Install libraries and configure keys
2. **RAG Pipeline** — Load Wiki and build vector database
3. **Router Agent** — Classify requests with Pydantic
4. **Tool Calling** — Call create_it_ticket
5. **Langfuse Tracing** — Monitor every step
6. **DeepEval** — Measure answer quality

In [ ]:
!pip install litellm pydantic deepeval chromadb sentence-transformers PyPDF2 langfuse -q

In [ ]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get('LANGFUSE_PUBLIC_KEY')
os.environ["LANGFUSE_SECRET_KEY"] = userdata.get('LANGFUSE_SECRET_KEY')
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"

print("✅ Keys loaded successfully")

---
##RAG

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

print("⏳ Loading Embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding model loaded")

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="hr_it_wiki",
)
print("✅ ChromaDB created")

In [ ]:
from google.colab import files
print("Upload IT_HR_Wiki.txt")
uploaded = files.upload()
print("Upload Employee_Queries.json")
uploaded = files.upload()

In [ ]:
with open("IT_HR_Wiki (For RAG DB).txt", "r") as f:
    wiki_text = f.read()

print(f"✅ File loaded: {len(wiki_text)} characters")
print(wiki_text[:100])

chunks = [c.strip() for c in wiki_text.split("\n\n") if c.strip()]

print(f"📄 Number of chunked paragraphs: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\n--- Paragraph {i+1} ---")
    print(chunk[:100] + "..." if len(chunk) > 100 else chunk)

In [ ]:
for i, chunk in enumerate(chunks):
    embedding = embedding_model.encode(chunk).tolist()

    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[f"chunk_{i}"]
    )
    print(f"✅ Paragraph stored {i+1}")

print(f"\n🎉 RAG built successfully! {len(chunks)} paragraphs in the database")

In [ ]:
def search_wiki(query: str, n_results: int = 1) -> str:
    """
    Searches Wiki for the closest paragraph to the query.

    query: The question the employee wants answered
    n_results: How many paragraphs to return (1 is usually enough)

    Returns: The closest text from Wiki
    """

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

    return results['documents'][0][0]

print("RAG Test:")
print("Question: VPN keeps disconnecting")
print("Score:", search_wiki("VPN keeps disconnecting"))

---
##Router Agent

In [ ]:
import litellm
import json
from pydantic import BaseModel
from typing import Literal

class TriageDecision(BaseModel):
    category: Literal["IT", "HR", "General"]
    requires_escalation: bool
    reason: str

print("✅ TriageDecision defined")
print("Required fields:")
print("  - category: IT or HR or General")
print("  - requires_escalation: True or False")
print("  - reason: reason for decision")

In [ ]:
def route_query(user_query: str) -> TriageDecision:
    import json

    system_prompt = """You are an IT/HR helpdesk router.
Classify employee requests using these strict rules:

category rules:
- "IT": technical issues (VPN, software, hardware, network)
- "HR": human resources (remote work, leave, payroll, policies)
- "General": questions unrelated to the company

requires_escalation rules (IMPORTANT):
- True ONLY for: physical hardware damage (broken keyboard, spilled liquid, damaged screen)
- False for: software issues that can be solved by instructions
- False for: HR policy questions
- False for: General questions

Respond ONLY with valid JSON like this:
{"category": "IT", "requires_escalation": false, "reason": "your reason here"}"""

    response = litellm.completion(
        model="openrouter/nvidia/nemotron-3-super-120b-a12b:free",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        temperature=0.1
    )

    content = response.choices[0].message.content
    content = content.strip().replace("```json", "").replace("```", "").strip()
    return TriageDecision(**json.loads(content))

print("Router Test:")
test = route_query("My VPN keeps disconnecting")
print(f"Category: {test.category}")
print(f"requires escalation؟ {test.requires_escalation}")
print(f"Reason: {test.reason}")

---
##Tool Calling

In [ ]:
create_ticket_tool = {
    "type": "function",
    "function": {
        "name": "create_it_ticket",
        "description": "Creates a formal IT support ticket for hardware damage or issues requiring physical intervention. Use ONLY for physical hardware problems.",
        "parameters": {
            "type": "object",
            "properties": {
                "issue_summary": {
                    "type": "string",
                    "description": "Brief description of the hardware issue"
                },
                "priority": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "Ticket priority level"
                },
                "category": {
                    "type": "string",
                    "enum": ["hardware", "software", "network"],
                    "description": "Type of IT issue"
                }
            },
            "required": ["issue_summary", "priority", "category"]
        }
    }
}

print("✅ create_it_ticket tool defined")

In [ ]:
import uuid
from datetime import datetime

def create_it_ticket(issue_summary: str, priority: str, category: str) -> dict:
    """
    Opens a formal IT ticket in the system.

    Returns: dict containing ticket number and creation confirmation
    """

    ticket_id = f"TKT-{uuid.uuid4().hex[:6].upper()}"

    ticket = {
        "success": True,
        "ticket_id": ticket_id,
        "issue_summary": issue_summary,
        "priority": priority,
        "category": category,
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "message": f"Ticket {ticket_id} created. IT team will contact you within 4 hours."
    }

    print(f"Ticket opened: {ticket_id}")
    return ticket


print("Function test:")
result = create_it_ticket("Keyboard damaged", "medium", "hardware")
print(result)

In [ ]:
def handle_with_tool_calling(user_query: str) -> dict:
    """
    Handles a request that needs a ticket opened using Tool Calling.

    Returns: dict containing the final answer and ticket information
    """

    print("📤 API Call 1: Sending question with tools list...")

    messages = [
        {
            "role": "system",
            "content": "You are an IT helpdesk assistant. For hardware damage, use create_it_ticket."
        },
        {
            "role": "user",
            "content": user_query
        }
    ]

    response_1 = litellm.completion(
        model="openrouter/nvidia/nemotron-3-super-120b-a12b:free",
        messages=messages,
        tools=[create_ticket_tool],
        tool_choice="auto",
        temperature=0.1
    )

    assistant_message = response_1.choices[0].message

    if assistant_message.tool_calls:
        print("Model requested a tool call!")

        tool_call = assistant_message.tool_calls[0]
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)

        print(f"   Tool: {tool_name}")
        print(f"   Parameters: {tool_args}")


        ticket_result = create_it_ticket(**tool_args)

        messages.append(assistant_message)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(ticket_result)
        })

        print("📤 API Call 2: Sending tool result to model...")

        response_2 = litellm.completion(
            model="openrouter/nvidia/nemotron-3-super-120b-a12b:free",
            messages=messages
        )

        final_answer = response_2.choices[0].message.content

        return {
            "answer": final_answer,
            "ticket_created": True,
            "ticket_id": ticket_result["ticket_id"],
            "tool_calls": [tool_name]
        }

    else:
        return {
            "answer": assistant_message.content,
            "ticket_created": False,
            "tool_calls": []
        }

print("✅ handle_with_tool_calling function defined")

---
##Langfuse


In [ ]:
import litellm
from langfuse import observe
from langfuse import observe

@observe()
def helpdesk_agent(user_query: str) -> dict:
    """
    The main helpdesk agent.

    Goes through three stages:
    1. Router: Classifies the question
    2. إذا requires escalation → Tool Calling
    3. If no escalation → RAG + answer
    """

    print(f"\n{'='*50}")
    print(f"📨 Question: {user_query}")
    print(f"{'='*50}")

    print("\n🎯 Stage 1: Classifying the question...")
    decision = route_query(user_query)
    print(f"   Category: {decision.category}")
    print(f"   requires escalation؟ {decision.requires_escalation}")
    print(f"   Reason: {decision.reason}")

    if decision.requires_escalation:
        print("\n🔴 Decision: Escalate → Open ticket")
        result = handle_with_tool_calling(user_query)
        result["category"] = decision.category
        result["escalated"] = True

    elif decision.category == "General":
        print("\n⚪ Decision: General question → Direct answer")
        result = {
            "answer": "This question is outside the scope of IT/HR support. Please consult other resources.",
            "category": "General",
            "escalated": False,
            "tool_calls": []
        }

    else:
        print("\n🟢 Decision: No escalation → Search Wiki")

        context = search_wiki(user_query)
        print(f"   Retrieved context: {context[:80]}...")

        rag_prompt = f"""Answer the employee's question using ONLY the following company policy.
Be specific and actionable.

Company Policy:
{context}

Employee Question: {user_query}"""

        rag_response = litellm.completion(
            model="openrouter/nvidia/nemotron-3-super-120b-a12b:free",
            messages=[{"role": "user", "content": rag_prompt}],
            temperature=0.1
        )

        result = {
            "answer": rag_response.choices[0].message.content,
            "category": decision.category,
            "escalated": False,
            "rag_context": context,
            "tool_calls": []
        }

    print(f"\n💬 Final Answer:\n{result['answer']}")
    return result

print("✅ helpdesk_agent defined")

---
## Running the Questions

Now we test the system on all questions from `Employee_Queries.json`

In [ ]:
import json

with open("Employee_Queries (For testing the agent).json", "r") as f:
    employee_queries = json.load(f)

print(f"✅ File loaded: {len(employee_queries)} questions")
print(employee_queries)

all_results = {}

for item in employee_queries:
    query_id = item["id"]
    query_text = item["query"]

    result = helpdesk_agent(query_text)
    all_results[query_id] = result

    if "rag_context" in result:
        all_results[query_id]["retrieval_context"] = [result["rag_context"]]

print("\n" + "="*50)
print("✅ All questions processed!")

---
## DeepEval


In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

print("✅ DeepEval initialized")

In [ ]:
import os
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import LiteLLMModel

judge_model = LiteLLMModel(
    model="openrouter/nvidia/nemotron-3-super-120b-a12b:free",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
)

print("Test 1: AnswerRelevancyMetric for VPN question")
print("-" * 40)

query_1 = employee_queries[0]["query"]
answer_1 = all_results[1]["answer"]
context_1 = all_results[1].get("retrieval_context", [])

test_case_1 = LLMTestCase(
    input=query_1,
    actual_output=answer_1,
    retrieval_context=context_1
)

relevancy_metric = AnswerRelevancyMetric(
    threshold=0.7,
    model=judge_model
)

relevancy_metric.measure(test_case_1)

print(f"Score: {relevancy_metric.score:.2f}")
print(f"Pass/Fail: {'✅ Passed' if relevancy_metric.is_successful() else '❌ Failed'}")
print(f"Reason: {relevancy_metric.reason}")

In [ ]:
print("Test 2: Verifying create_it_ticket was called")
print("-" * 40)

query_2 = employee_queries[1]["query"]
result_2 = all_results[2]

print(f"Question: {query_2}")
print(f"Ticket created? {result_2.get('ticket_created', False)}")
print(f"tool_calls list: {result_2.get('tool_calls', [])}")

assert "create_it_ticket" in result_2.get("tool_calls", []), \
    "❌ Error: create_it_ticket was not called! Make sure the Router escalates broken hardware requests."

print("\n✅ Test 2 passed: create_it_ticket was called correctly!")
if "ticket_id" in result_2:
    print(f"Ticket ID: {result_2['ticket_id']}")

In [ ]:
# ══════════════════════════════════════════════════════
# Final summary of all results
# ══════════════════════════════════════════════════════
print("\n" + "="*60)
print("Summary of all question results")
print("="*60)

expected = [
    {"id": 1, "expected_category": "IT",      "expected_escalation": False, "expected_action": "RAG"},
    {"id": 2, "expected_category": "IT",      "expected_escalation": True,  "expected_action": "Ticket"},
    {"id": 3, "expected_category": "HR",      "expected_escalation": False, "expected_action": "RAG"},
    {"id": 4, "expected_category": "General", "expected_escalation": False, "expected_action": "General"}
]

for item in employee_queries:
    qid = item["id"]
    result = all_results[qid]
    exp = expected[qid - 1]

    actual_escalation = result.get("escalated", result.get("ticket_created", False))
    escalation_correct = actual_escalation == exp["expected_escalation"]

    status = "✅" if escalation_correct else "❌"

    print(f"\n{status} questions {qid}: {item['query'][:50]}...")
    print(f"   Category: {result.get('category', 'N/A')} | Escalation: {actual_escalation}")
    print(f"   Answer: {result['answer'][:100]}...")

print("\n" + "="*60)
print("Project ran successfully!")
print("="*60)